# Experimentos de rendimiento — WikiRank

Mediciones que se comentan en la memoria. Requiere haber ejecutado antes `pipeline.ipynb` (lee `/curated`).

| Celda | Experimento | Memoria |
|---|---|---|
| MapReduce | Media por lengua con Hadoop Streaming (ejecutado en terminal) y comprobación frente a P1 | 4.1, 4.3 |
| RDD | Media por lengua con RDDs (comparación con MapReduce) | 4.2–4.3 |
| E4 | Misma consulta sobre CSV y sobre Parquet | 3.4 |
| E5–E6 | Join de P3b: sort-merge frente a broadcast | 5.3 |
| E7–E8 | Data skew y AQE (desafío F.3) | 7 |

In [ ]:
import os, time, yaml
from pyspark.sql import SparkSession, Window, functions as F

with open("/home/jovyan/work/config.yaml") as f:
    cfg = yaml.safe_load(f)

HDFS = cfg["hdfs_base"]
RAW = HDFS + cfg["paths"]["raw"]
CURATED = HDFS + cfg["paths"]["curated"]
LANG_INFO = HDFS + cfg["paths"]["language_info"]
RESULTS = HDFS + cfg["paths"]["results"]

os.environ["HADOOP_USER_NAME"] = "root"

spark = (SparkSession.builder
         .appName("wikirank-experiments")
         .master(cfg["spark"]["master"])
         .config("spark.driver.host", cfg["spark"]["driver_host"])
         .config("spark.executor.memory", cfg["spark"]["executor_memory"])
         .config("spark.hadoop.dfs.replication", cfg["spark"]["dfs_replication"])
         .getOrCreate())
spark


In [ ]:
articles = spark.read.parquet(CURATED)
lang_info = spark.read.parquet(LANG_INFO)
RAW_FILE = RAW + "/" + os.path.basename(cfg["input"]["local_csv"])
schema = ("Wikipedia_article_URL STRING, Language STRING, Page_ID LONG, "
          "Title STRING, WikiRank_score DOUBLE, WikiRank_URL STRING")
raw = spark.read.schema(schema).option("header", True).option("escape", '"').csv(RAW_FILE)

## MapReduce: número de artículos y media por lengua (Hadoop Streaming)

El job no se puede lanzar desde Jupyter (el comando `hadoop` solo existe en los contenedores de Hadoop), así que se ejecutó en una terminal, sobre el **mismo CSV** de la zona raw que usa la versión con RDDs. Requiere `mapper.py` y `reducer.py` (carpeta `mapreduce/`) y Python 3 en el NodeManager (ver README).

```powershell
docker exec -it namenode hdfs dfs -rm -r -f /results/mr_stats_by_language
docker exec -it namenode bash -c "time hadoop jar /opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar -files /mapreduce/mapper.py,/mapreduce/reducer.py -mapper 'python3 mapper.py' -reducer 'python3 reducer.py' -input /raw/wikirank -output /results/mr_stats_by_language"
```

La primera línea borra la salida anterior: Hadoop no escribe en una carpeta que ya existe, así que el job es repetible.

**Tiempo:** `real 11m19.505s` (679 s). El job terminó con estado SUCCEEDED en YARN (captura en `evidencias/`).

**Salida** (`hdfs dfs -cat /results/mr_stats_by_language/part-00000`), primeras líneas, con lengua, artículos y media:
```
ar      1237943 17.109
az      201054  15.343
be      244985  12.946
bg      299787  17.745
ca      756348  21.161
...
en      6860430 24.712
es      1905093 18.765
...
zh-min-nan      433220  21.541
```
55 lenguas, 47.019.883 artículos. La celda siguiente comprueba que coinciden con P1.

In [ ]:
MR_OUT = RESULTS + "/mr_stats_by_language"
mr = (spark.read.option("sep", "\t")
        .schema("Language STRING, n_articles LONG, avg_score DOUBLE")
        .csv(MR_OUT))
p1 = spark.read.parquet(RESULTS + "/q1_stats_by_language")

cmp = (mr.withColumnRenamed("n_articles", "n_mr").withColumnRenamed("avg_score", "avg_mr")
         .join(p1, "Language", "full_outer"))
diff = cmp.filter(F.col("n_mr").isNull() | F.col("n_articles").isNull()
                  | (F.col("n_mr") != F.col("n_articles"))
                  | (F.abs(F.col("avg_mr") - F.col("avg_score")) > 0.01))

print("Lenguas en MapReduce:", mr.count(), "| artículos:", mr.agg(F.sum("n_articles")).first()[0])
print("Lenguas que no coinciden con P1:", diff.count())
diff.show()

In [ ]:
import csv

lines = spark.sparkContext.textFile(RAW_FILE)
header = lines.first()

def parse(line):
    f = next(csv.reader([line]))
    try:
        lang, score = f[1].strip().lower(), float(f[4])
    except (IndexError, ValueError):
        return []
    return [(lang, score)] if lang and 0 <= score <= 100 else []

t = time.time()
result = (lines.filter(lambda l: l != header)
               .flatMap(parse)
               .mapValues(lambda s: (s, 1))
               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
               .mapValues(lambda x: (x[1], round(x[0] / x[1], 2)))
               .sortBy(lambda kv: -kv[1][0])
               .collect())
print(f"RDD: {time.time() - t:.1f} s")
for lang, (n, avg) in result[:10]:
    print(lang, n, avg)


RDD: 66.2 s
en 6860430 24.71
de 2930971 18.63
fr 2626210 16.12
sv 2591505 20.02
nl 2163715 12.73
ru 1991787 18.18
es 1905093 18.77
it 1875385 13.63
pl 1624175 17.03
zh 1433026 20.31


**Optimización**

In [ ]:
t = time.time()
raw.agg(F.avg("WikiRank_score")).show()
print(f"CSV:     {time.time() - t:.1f} s")

t = time.time()
articles.agg(F.avg("WikiRank_score")).show()
print(f"Parquet: {time.time() - t:.1f} s")

articles.agg(F.avg("WikiRank_score")).explain()

+-------------------+
|avg(WikiRank_score)|
+-------------------+
| 17.932223616843302|
+-------------------+

CSV:     30.7 s
+-------------------+
|avg(WikiRank_score)|
+-------------------+
| 17.932223616844187|
+-------------------+

Parquet: 1.5 s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(WikiRank_score#2)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=92]
      +- HashAggregate(keys=[], functions=[partial_avg(WikiRank_score#2)])
         +- Project [WikiRank_score#2]
            +- FileScan parquet [WikiRank_score#2,Language#4] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<WikiRank_score:double>




In [ ]:
def q3_expr(info_table):
    return (articles.join(info_table, "Language", "left")
                    .groupBy("script")
                    .agg(F.count("*").alias("n_articles"),
                         F.percentile_approx("WikiRank_score", 0.5, 10000).alias("median_score")))

def run(q):
    """Ejecuta la consulta completa sin guardar nada (noop) y devuelve el tiempo en segundos."""
    t = time.time()
    q.write.format("noop").mode("overwrite").save()
    return time.time() - t

lang_sub = lang_info.select("Language", "script")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
q_smj = q3_expr(lang_sub)
q_bc = q3_expr(F.broadcast(lang_sub))

q_smj.explain()
q_bc.explain()

run(q_bc)
times = {"Sort-merge join": [], "Broadcast join": []}
for _ in range(3):
    times["Sort-merge join"].append(run(q_smj))
    times["Broadcast join"].append(run(q_bc))
for name, v in times.items():
    print(f"{name}: {', '.join(f'{x:.1f}' for x in v)} s | mediana {sorted(v)[1]:.1f} s")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ObjectHashAggregate(keys=[script#8], functions=[count(1), percentile_approx(WikiRank_score#2, 0.5, 10000, 0, 0)])
   +- Exchange hashpartitioning(script#8, 200), ENSURE_REQUIREMENTS, [plan_id=134]
      +- ObjectHashAggregate(keys=[script#8], functions=[partial_count(1), partial_percentile_approx(WikiRank_score#2, 0.5, 10000, 0, 0)])
         +- Project [WikiRank_score#2, script#8]
            +- SortMergeJoin [Language#4], [Language#5], LeftOuter
               :- Sort [Language#4 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(Language#4, 200), ENSURE_REQUIREMENTS, [plan_id=126]
               :     +- FileScan parquet [WikiRank_score#2,Language#4] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<WikiRank_score:double>
               +- Sort [Language#5 ASC NULLS FI

In [ ]:
counts = articles.groupBy("Language").count().orderBy(F.desc("count"))
counts.show(5)
median = counts.approxQuantile("count", [0.5], 0.0)[0]
top = counts.first()["count"]
print(f"Mayor: {top} | mediana: {median:.0f} | {top / median:.1f} veces la mediana")

+--------+-------+
|Language|  count|
+--------+-------+
|      en|6860430|
|      de|2930971|
|      fr|2626210|
|      sv|2591505|
|      nl|2163715|
+--------+-------+
only showing top 5 rows
Mayor: 6860430 | mediana: 433220 | 15.8 veces la mediana


In [ ]:
avg_by_lang = articles.groupBy("Language").agg(F.avg("WikiRank_score").alias("avg_score"))
q_skew = articles.join(avg_by_lang, "Language")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 3)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "32m")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "16m")

def run_aqe(enabled):
    spark.conf.set("spark.sql.adaptive.enabled", enabled)
    return run(q_skew)

run_aqe(True)
times = {"Sin AQE": [], "Con AQE + skewJoin": []}
for _ in range(3):
    times["Sin AQE"].append(run_aqe(False))
    times["Con AQE + skewJoin"].append(run_aqe(True))
for name, v in times.items():
    print(f"{name}: {', '.join(f'{x:.1f}' for x in v)} s | mediana {sorted(v)[1]:.1f} s")

spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

Sin AQE: 31.8 s
Con AQE: 32.3 s


In [9]:
#spark.stop()